In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
INPUT_FILE = Path(
    "data/idx_financial_period_mapped_corrected.csv"
)

OUTPUT_FILE = Path(
    "data/idx_financial_current_metrics_selected.csv"
)

CONFLICT_FILE = Path(
    "data/idx_financial_current_metric_conflicts.csv"
)

print("Input exists:", INPUT_FILE.exists())
print("Input file:", INPUT_FILE)

Input exists: True
Input file: data\idx_financial_period_mapped_corrected.csv


In [3]:
mapped_df = pd.read_csv(
    INPUT_FILE
)

print("Rows:", len(mapped_df))
print("Columns:", mapped_df.columns.tolist())

display(
    mapped_df.head()
)

Rows: 205033
Columns: ['ticker', 'year', 'quarter', 'metric', 'source_label', 'source_sheet', 'row_number', 'source_file', 'source_path', 'candidate_column', 'candidate_value', 'period_type', 'period_header', 'period_header_row', 'mapping_status', 'parsed_period_date', 'unique_dates', 'latest_date', 'earliest_date', 'expected_period_date']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_30008\3742055099.py:1: DtypeWarning: Columns (0: source_sheet, 1: parsed_period_date, 2: latest_date, 3: earliest_date) have mixed types. Specify dtype option on import or set low_memory=False.
  mapped_df = pd.read_csv(


,ticker,year,quarter,metric,source_label,source_sheet,row_number,source_file,source_path,candidate_column,candidate_value,period_type,period_header,period_header_row,mapping_status,parsed_period_date,unique_dates,latest_date,earliest_date,expected_period_date
0,ZYRX,2025,Q1,cash,Kas dan setara kas,1210000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,1.888963e+09,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
1,ZYRX,2025,Q1,cash,Kas dan setara kas,1210000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,2.0,6.272314e+09,COMPARATIVE,PriorEndYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
2,ZYRX,2025,Q1,total_assets,Jumlah aset,1210000,128,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,3.964298e+11,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
3,ZYRX,2025,Q1,total_assets,Jumlah aset,1210000,128,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,2.0,3.924446e+11,COMPARATIVE,PriorEndYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
4,ZYRX,2025,Q1,total_liabilities,Jumlah liabilitas,1210000,247,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,9.811678e+10,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31


In [5]:
period_summary = (
    mapped_df["period_type"]
    .value_counts(
        dropna=False
    )
)

display(
    period_summary
)

unknown_count = (
    mapped_df["period_type"]
    .eq("UNKNOWN")
    .sum()
)

print(
    "UNKNOWN rows:",
    unknown_count
)

assert unknown_count == 0, (
    "Masih ada UNKNOWN period. "
    "Jangan lanjut final selection."
)

period_type
COMPARATIVE      89731
CURRENT          89709
VALUE_MISSING    25593
Name: count, dtype: int64

UNKNOWN rows: 0


In [6]:
current_df = (
    mapped_df[
        mapped_df["period_type"]
        .eq("CURRENT")
    ]
    .copy()
)

print(
    "CURRENT rows:",
    len(current_df)
)

display(
    current_df.head()
)

CURRENT rows: 89709


,ticker,year,quarter,metric,source_label,source_sheet,row_number,source_file,source_path,candidate_column,candidate_value,period_type,period_header,period_header_row,mapping_status,parsed_period_date,unique_dates,latest_date,earliest_date,expected_period_date
0,ZYRX,2025,Q1,cash,Kas dan setara kas,1210000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,1.888963e+09,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
2,ZYRX,2025,Q1,total_assets,Jumlah aset,1210000,128,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,3.964298e+11,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
4,ZYRX,2025,Q1,total_liabilities,Jumlah liabilitas,1210000,247,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,9.811678e+10,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
6,ZYRX,2025,Q1,revenue,Sales and revenue,1321000,6,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,4.338110e+10,CURRENT,CurrentYearDuration,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31
11,ZYRX,2025,Q1,gross_profit,Jumlah laba bruto,1321000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,9.238585e+09,CURRENT,CurrentYearDuration,4.0,EXPLICIT_TOKEN,NaN,NaN,NaN,NaN,2025-03-31


In [7]:
current_df[
    "candidate_value"
] = pd.to_numeric(
    current_df[
        "candidate_value"
    ],
    errors="coerce"
)

print(
    "CURRENT rows with missing candidate value:",
    current_df[
        "candidate_value"
    ].isna().sum()
)

CURRENT rows with missing candidate value: 0


In [8]:
GROUP_KEYS = [
    "ticker",
    "year",
    "quarter",
    "metric"
]

In [9]:
report_file_count = (
    current_df
    .groupby(
        [
            "ticker",
            "year",
            "quarter"
        ]
    )["source_file"]
    .nunique()
    .reset_index(
        name="source_file_count"
    )
)

multiple_file_reports = (
    report_file_count[
        report_file_count[
            "source_file_count"
        ] > 1
    ]
)

print(
    "Ticker-period with more than 1 source file:",
    len(multiple_file_reports)
)

display(
    multiple_file_reports.head(100)
)

Ticker-period with more than 1 source file: 125


,ticker,year,quarter,source_file_count
0,AADI,2024,Q4,2
1346,AVIA,2021,Q4,2
1350,AVIA,2022,Q4,2
2066,BFIN,2021,Q4,2
2070,BFIN,2022,Q4,2
...,...,...,...,...
13301,SMRU,2021,Q2,2
13302,SMRU,2021,Q3,2
13307,SMRU,2022,Q4,2
13472,SPMA,2021,Q2,2


In [10]:
candidate_summary_df = (
    current_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        current_candidate_count=(
            "candidate_value",
            "size"
        ),

        unique_current_values=(
            "candidate_value",
            "nunique"
        ),

        source_file_count=(
            "source_file",
            "nunique"
        ),

        source_sheet_count=(
            "source_sheet",
            "nunique"
        )
    )
    .reset_index()
)

print(
    "Metric groups:",
    len(candidate_summary_df)
)

display(
    candidate_summary_df.head()
)

Metric groups: 88916


,ticker,year,quarter,metric,current_candidate_count,unique_current_values,source_file_count,source_sheet_count
0,AADI,2024,Q4,cash,2,1,2,1
1,AADI,2024,Q4,gross_profit,2,1,2,1
2,AADI,2024,Q4,operating_cash_flow,2,1,2,1
3,AADI,2024,Q4,revenue,2,1,2,1
4,AADI,2024,Q4,total_assets,2,1,2,1


In [11]:
unique_value_distribution = (
    candidate_summary_df[
        "unique_current_values"
    ]
    .value_counts()
    .sort_index()
)

display(
    unique_value_distribution
)

unique_current_values
1    88371
2      543
5        1
6        1
Name: count, dtype: int64

In [12]:
unique_current_df = (
    current_df
    .drop_duplicates(
        subset=
        GROUP_KEYS
        +
        [
            "candidate_value"
        ]
    )
    .copy()
)

print(
    "CURRENT rows before value dedupe:",
    len(current_df)
)

print(
    "CURRENT rows after value dedupe:",
    len(unique_current_df)
)

CURRENT rows before value dedupe: 89709
CURRENT rows after value dedupe: 89468


In [13]:
safe_groups_df = (
    candidate_summary_df[
        candidate_summary_df[
            "unique_current_values"
        ].eq(1)
    ]
    .copy()
)

conflict_groups_df = (
    candidate_summary_df[
        candidate_summary_df[
            "unique_current_values"
        ] > 1
    ]
    .copy()
)

print(
    "Safe groups:",
    len(safe_groups_df)
)

print(
    "Conflict groups:",
    len(conflict_groups_df)
)

Safe groups: 88371
Conflict groups: 545


In [14]:
selected_current_df = (
    unique_current_df
    .merge(
        safe_groups_df[
            GROUP_KEYS
            +
            [
                "current_candidate_count",
                "unique_current_values",
                "source_file_count",
                "source_sheet_count"
            ]
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .drop_duplicates(
        subset=GROUP_KEYS
    )
    .copy()
)

In [16]:
selected_current_df[
    "selection_status"
] = np.where(
    selected_current_df[
        "current_candidate_count"
    ].eq(1),

    "SELECTED_SINGLE_CANDIDATE",

    "SELECTED_MULTIPLE_SAME_VALUE"
)

selected_current_df = (
    selected_current_df
    .rename(
        columns={
            "candidate_value":
                "selected_value"
        }
    )
)

In [17]:
FINAL_COLUMNS = [
    "ticker",
    "year",
    "quarter",
    "metric",

    "selected_value",

    "selection_status",

    "current_candidate_count",
    "unique_current_values",

    "source_file_count",
    "source_sheet_count",

    "source_label",
    "source_sheet",
    "row_number",
    "source_file",
    "source_path",

    "period_header",
    "period_header_row",
    "mapping_status"
]

selected_current_df = (
    selected_current_df[
        FINAL_COLUMNS
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    selected_current_df.head(100)
)

,ticker,year,quarter,metric,selected_value,selection_status,current_candidate_count,unique_current_values,source_file_count,source_sheet_count,source_label,source_sheet,row_number,source_file,source_path,period_header,period_header_row,mapping_status
0,AADI,2024,Q4,cash,1518688.0,SELECTED_MULTIPLE_SAME_VALUE,2,1,2,1,Kas dan setara kas,1210000,8,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,CurrentYearInstant,4.0,EXPLICIT_TOKEN
1,AADI,2024,Q4,gross_profit,1465951.0,SELECTED_MULTIPLE_SAME_VALUE,2,1,2,1,Jumlah laba bruto,1321000,8,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,CurrentYearDuration,4.0,EXPLICIT_TOKEN
2,AADI,2024,Q4,operating_cash_flow,1198515.0,SELECTED_MULTIPLE_SAME_VALUE,2,1,2,1,Total net cash flows received from (used in) o...,1510000,47,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,CurrentYearDuration,4.0,EXPLICIT_TOKEN
3,AADI,2024,Q4,revenue,5319582.0,SELECTED_MULTIPLE_SAME_VALUE,2,1,2,1,Sales and revenue,1321000,6,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,CurrentYearDuration,4.0,EXPLICIT_TOKEN
4,AADI,2024,Q4,total_assets,5992658.0,SELECTED_MULTIPLE_SAME_VALUE,2,1,2,1,Jumlah aset,1210000,128,AADI_2024_Q4_FinancialStatement-2024-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,CurrentYearInstant,4.0,EXPLICIT_TOKEN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,AALI,2023,Q2,total_liabilities,7229331.0,SELECTED_SINGLE_CANDIDATE,1,1,1,1,Jumlah liabilitas,1210000,246,AALI_2023_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,30 June 2023,4.0,DATE_HEADER_PAIR
96,AALI,2023,Q3,cash,2522256.0,SELECTED_SINGLE_CANDIDATE,1,1,1,1,Kas dan setara kas,1210000,7,AALI_2023_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,30 September 2023,4.0,DATE_HEADER_PAIR
97,AALI,2023,Q3,gross_profit,1949932.0,SELECTED_SINGLE_CANDIDATE,1,1,1,1,Jumlah laba bruto,1321000,7,AALI_2023_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,30 September 2023,4.0,DATE_HEADER_PAIR
98,AALI,2023,Q3,operating_cash_flow,2270148.0,SELECTED_SINGLE_CANDIDATE,1,1,1,1,Total net cash flows received from (used in) o...,1510000,46,AALI_2023_Q3_FS.xlsx,data\idx_financial_statements\Financial_Statem...,30 September 2023,4.0,DATE_HEADER_PAIR


In [18]:
conflict_detail_df = (
    current_df
    .merge(
        conflict_groups_df[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

conflict_detail_df = (
    conflict_detail_df
    .sort_values(
        GROUP_KEYS
        +
        [
            "candidate_value"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Conflict detail rows:",
    len(conflict_detail_df)
)

display(
    conflict_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_label",
            "source_sheet",
            "row_number",
            "source_file"
        ]
    ].head(200)
)

Conflict detail rows: 1110


,ticker,year,quarter,metric,candidate_value,source_label,source_sheet,row_number,source_file
0,ASHA,2024,Q2,revenue,2.317741e+09,PENJUALAN,1617000,20,ASHA_2024_Q2_FS.xlsx
1,ASHA,2024,Q2,revenue,2.317741e+09,PENJUALAN,1618000,8,ASHA_2024_Q2_FS.xlsx
2,ASHA,2024,Q2,revenue,2.052546e+10,PENJUALAN,1618000,19,ASHA_2024_Q2_FS.xlsx
3,ASHA,2024,Q2,revenue,2.052546e+10,PENJUALAN,1618000,30,ASHA_2024_Q2_FS.xlsx
4,ASHA,2024,Q2,revenue,8.265174e+10,PENJUALAN,1618000,7,ASHA_2024_Q2_FS.xlsx
...,...,...,...,...,...,...,...,...,...
195,CTTH,2021,Q4,operating_cash_flow,4.895119e+06,Total net cash flows received from (used in) o...,1510000,37,CTTH_2021_Q4_FS.xlsx
196,CTTH,2021,Q4,revenue,2.432205e+07,Sales and revenue,1321000,5,CTTH_2021_Q4_FS.xlsx
197,CTTH,2021,Q4,revenue,1.687930e+11,Sales and revenue,1311000,5,CTTH_2021_Q4_FinancialStatement-2021-Tahunan-A...
198,CTTH,2021,Q4,total_assets,3.039991e+07,Jumlah aset,1210000,123,CTTH_2021_Q4_FS.xlsx


In [20]:
selection_summary = (
    selected_current_df[
        "selection_status"
    ]
    .value_counts()
    .reset_index()
)

selection_summary.columns = [
    "selection_status",
    "rows"
]

display(
    selection_summary
)

metric_selection_summary = (
    selected_current_df
    .groupby(
        "metric"
    )
    .agg(
        selected_rows=(
            "metric",
            "size"
        ),

        tickers=(
            "ticker",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "selected_rows",
        ascending=False
    )
)

display(
    metric_selection_summary
)

,selection_status,rows
0,SELECTED_SINGLE_CANDIDATE,88143
1,SELECTED_MULTIPLE_SAME_VALUE,228


,metric,selected_rows,tickers
4,total_assets,15598,948
5,total_liabilities,15598,948
2,operating_cash_flow,15582,948
0,cash,14587,900
3,revenue,13799,867
1,gross_profit,13207,834


In [21]:
conflict_summary = (
    conflict_groups_df
    .groupby(
        "metric"
    )
    .size()
    .reset_index(
        name="conflict_groups"
    )
    .sort_values(
        "conflict_groups",
        ascending=False
    )
)

display(
    conflict_summary
)

,metric,conflict_groups
3,revenue,101
0,cash,89
4,total_assets,89
2,operating_cash_flow,89
5,total_liabilities,89
1,gross_profit,88


In [26]:
conflict_audit_df = (
    conflict_detail_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        conflict_rows=(
            "candidate_value",
            "size"
        ),

        unique_values=(
            "candidate_value",
            "nunique"
        ),

        source_files=(
            "source_file",
            "nunique"
        ),

        source_sheets=(
            "source_sheet",
            "nunique"
        ),

        source_labels=(
            "source_label",
            "nunique"
        )
    )
    .reset_index()
)

print(
    "Conflict groups:",
    len(conflict_audit_df)
)

display(
    conflict_audit_df.head(100)
)

Conflict groups: 545


,ticker,year,quarter,metric,conflict_rows,unique_values,source_files,source_sheets,source_labels
0,ASHA,2024,Q2,revenue,11,6,1,3,2
1,ASHA,2024,Q3,revenue,5,5,1,3,2
2,AVIA,2021,Q4,cash,2,2,2,1,1
3,AVIA,2021,Q4,gross_profit,2,2,2,2,1
4,AVIA,2021,Q4,operating_cash_flow,2,2,2,1,1
...,...,...,...,...,...,...,...,...,...
95,CTTH,2022,Q4,cash,2,2,2,1,1
96,CTTH,2022,Q4,gross_profit,2,2,2,2,1
97,CTTH,2022,Q4,operating_cash_flow,2,2,2,1,1
98,CTTH,2022,Q4,revenue,2,2,2,2,1


In [27]:
conflict_pattern_summary = (
    conflict_audit_df
    .groupby(
        [
            "unique_values",
            "source_files",
            "source_sheets",
            "source_labels"
        ]
    )
    .size()
    .reset_index(
        name="groups"
    )
    .sort_values(
        "groups",
        ascending=False
    )
)

display(
    conflict_pattern_summary.head(100)
)

,unique_values,source_files,source_sheets,source_labels,groups
4,2,2,1,1,351
5,2,2,2,1,174
1,2,1,2,2,9
6,2,3,1,1,4
7,2,3,2,1,2
0,2,1,2,1,1
2,2,1,3,2,1
3,2,1,3,3,1
8,5,1,3,2,1
9,6,1,3,2,1


In [28]:
multiple_source_file_conflicts = (
    conflict_audit_df[
        conflict_audit_df[
            "source_files"
        ] > 1
    ]
    .copy()
)

print(
    "Conflict groups with multiple source files:",
    len(multiple_source_file_conflicts)
)

display(
    multiple_source_file_conflicts.head(100)
)

Conflict groups with multiple source files: 531


,ticker,year,quarter,metric,conflict_rows,unique_values,source_files,source_sheets,source_labels
2,AVIA,2021,Q4,cash,2,2,2,1,1
3,AVIA,2021,Q4,gross_profit,2,2,2,2,1
4,AVIA,2021,Q4,operating_cash_flow,2,2,2,1,1
5,AVIA,2021,Q4,revenue,2,2,2,2,1
6,AVIA,2021,Q4,total_assets,2,2,2,1,1
...,...,...,...,...,...,...,...,...,...
97,CTTH,2022,Q4,operating_cash_flow,2,2,2,1,1
98,CTTH,2022,Q4,revenue,2,2,2,2,1
99,CTTH,2022,Q4,total_assets,2,2,2,1,1
100,CTTH,2022,Q4,total_liabilities,2,2,2,1,1


In [29]:
single_source_file_conflicts = (
    conflict_audit_df[
        conflict_audit_df[
            "source_files"
        ] == 1
    ]
    .copy()
)

print(
    "Conflict groups from a single source file:",
    len(single_source_file_conflicts)
)

Conflict groups from a single source file: 14


In [30]:
sample_conflict_groups = (
    conflict_audit_df
    .head(20)
    [GROUP_KEYS]
)

sample_conflict_detail_df = (
    conflict_detail_df
    .merge(
        sample_conflict_groups,
        on=GROUP_KEYS,
        how="inner"
    )
    .sort_values(
        GROUP_KEYS
        +
        [
            "candidate_value"
        ]
    )
)

display(
    sample_conflict_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_label",
            "source_sheet",
            "row_number",
            "source_file",
            "period_header",
            "mapping_status"
        ]
    ].head(200)
)

,ticker,year,quarter,metric,candidate_value,source_label,source_sheet,row_number,source_file,period_header,mapping_status
0,ASHA,2024,Q2,revenue,2.317741e+09,PENJUALAN,1617000,20,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
1,ASHA,2024,Q2,revenue,2.317741e+09,PENJUALAN,1618000,8,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
2,ASHA,2024,Q2,revenue,2.052546e+10,PENJUALAN,1618000,19,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
3,ASHA,2024,Q2,revenue,2.052546e+10,PENJUALAN,1618000,30,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
4,ASHA,2024,Q2,revenue,8.265174e+10,PENJUALAN,1618000,7,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
5,ASHA,2024,Q2,revenue,8.496948e+10,PENJUALAN,1618000,18,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
6,ASHA,2024,Q2,revenue,1.031772e+11,PENJUALAN,1617000,19,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
7,ASHA,2024,Q2,revenue,1.054949e+11,Sales and revenue,1311000,6,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
8,ASHA,2024,Q2,revenue,1.054949e+11,PENJUALAN,1617000,30,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN
9,ASHA,2024,Q2,revenue,1.054949e+11,PENJUALAN,1617000,31,ASHA_2024_Q2_FS.xlsx,CurrentYearDuration,EXPLICIT_TOKEN


In [31]:
conflict_year_summary = (
    conflict_audit_df
    .groupby(
        "year"
    )
    .size()
    .reset_index(
        name="conflict_groups"
    )
    .sort_values(
        "year"
    )
)

display(
    conflict_year_summary
)

,year,conflict_groups
0,2021,264
1,2022,266
2,2023,10
3,2024,3
4,2025,2


In [32]:
# CELL 23A
multi_file_conflict_detail = (
    conflict_detail_df
    .merge(
        multiple_source_file_conflicts[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

multi_file_conflict_file_summary = (
    multi_file_conflict_detail
    .groupby(
        [
            "ticker",
            "year",
            "quarter",
            "metric"
        ]
    )
    .agg(
        source_files=(
            "source_file",
            lambda x: sorted(set(x))
        ),

        values=(
            "candidate_value",
            lambda x: sorted(set(x))
        ),

        file_count=(
            "source_file",
            "nunique"
        )
    )
    .reset_index()
)

display(
    multi_file_conflict_file_summary.head(100)
)

,ticker,year,quarter,metric,source_files,values,file_count
0,AVIA,2021,Q4,cash,"[AVIA_2021_Q4_FS.xlsx, AVIA_2021_Q4_FinancialS...","[3896022.0, 18722391189.0]",2
1,AVIA,2021,Q4,gross_profit,"[AVIA_2021_Q4_FS.xlsx, AVIA_2021_Q4_FinancialS...","[4830014.0, 77326783295.0]",2
2,AVIA,2021,Q4,operating_cash_flow,"[AVIA_2021_Q4_FS.xlsx, AVIA_2021_Q4_FinancialS...","[-28394083403.0, 4895119.0]",2
3,AVIA,2021,Q4,revenue,"[AVIA_2021_Q4_FS.xlsx, AVIA_2021_Q4_FinancialS...","[24322048.0, 168792972085.0]",2
4,AVIA,2021,Q4,total_assets,"[AVIA_2021_Q4_FS.xlsx, AVIA_2021_Q4_FinancialS...","[30399906.0, 524632899688.0]",2
...,...,...,...,...,...,...,...
95,CTTH,2022,Q4,operating_cash_flow,"[CTTH_2022_Q4_FS.xlsx, CTTH_2022_Q4_FinancialS...","[1835397.0, 479691758.0]",2
96,CTTH,2022,Q4,revenue,"[CTTH_2022_Q4_FS.xlsx, CTTH_2022_Q4_FinancialS...","[21828591.0, 908142046.0]",2
97,CTTH,2022,Q4,total_assets,"[CTTH_2022_Q4_FS.xlsx, CTTH_2022_Q4_FinancialS...","[29249340.0, 1286624764.0]",2
98,CTTH,2022,Q4,total_liabilities,"[CTTH_2022_Q4_FS.xlsx, CTTH_2022_Q4_FinancialS...","[7006119.0, 717317140.0]",2


In [33]:
# CELL 23B
source_file_frequency = (
    multi_file_conflict_detail[
        "source_file"
    ]
    .value_counts()
    .reset_index()
)

source_file_frequency.columns = [
    "source_file",
    "conflict_rows"
]

display(
    source_file_frequency.head(100)
)

,source_file,conflict_rows
0,AVIA_2021_Q4_FS.xlsx,6
1,AVIA_2021_Q4_FinancialStatement-2021-Tahunan-A...,6
2,AVIA_2022_Q4_FS.xlsx,6
3,AVIA_2022_Q4_FinancialStatement-2022-Tahunan-A...,6
4,BFIN_2021_Q4_FS.xlsx,6
...,...,...
95,MEGA_2022_Q4_FinancialStatement-2022-Tahunan-A...,6
96,MERK_2021_Q4_FS.xlsx,6
97,MERK_2021_Q4_FinancialStatement-2021-Tahunan-A...,6
98,MERK_2022_Q4_FS.xlsx,6


In [34]:
# CELL 23C
sample_multi_file_conflicts = (
    multi_file_conflict_detail[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "source_sheet",
            "source_label",
            "row_number",
            "period_header",
            "mapping_status"
        ]
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_file"
        ]
    )
)

display(
    sample_multi_file_conflicts.head(200)
)

,ticker,year,quarter,metric,candidate_value,source_file,source_sheet,source_label,row_number,period_header,mapping_status
0,AVIA,2021,Q4,cash,3.896022e+06,AVIA_2021_Q4_FS.xlsx,1210000,Kas dan setara kas,7,31 December 2021,DATE_HEADER_PAIR
1,AVIA,2021,Q4,cash,1.872239e+10,AVIA_2021_Q4_FinancialStatement-2021-Tahunan-A...,1210000,Kas dan setara kas,7,31 December 2021,DATE_HEADER_PAIR
2,AVIA,2021,Q4,gross_profit,4.830014e+06,AVIA_2021_Q4_FS.xlsx,1321000,Jumlah laba bruto,7,31 December 2021,DATE_HEADER_PAIR
3,AVIA,2021,Q4,gross_profit,7.732678e+10,AVIA_2021_Q4_FinancialStatement-2021-Tahunan-A...,1311000,Jumlah laba bruto,7,31 December 2021,DATE_HEADER_PAIR
5,AVIA,2021,Q4,operating_cash_flow,4.895119e+06,AVIA_2021_Q4_FS.xlsx,1510000,Total net cash flows received from (used in) o...,37,31 December 2021,DATE_HEADER_PAIR
...,...,...,...,...,...,...,...,...,...,...,...
195,CTTH,2022,Q4,total_assets,1.286625e+09,CTTH_2022_Q4_FinancialStatement-2022-Tahunan-A...,1210000,Jumlah aset,127,31 December 2022,DATE_HEADER_PAIR
196,CTTH,2022,Q4,total_liabilities,7.006119e+06,CTTH_2022_Q4_FS.xlsx,1210000,Jumlah liabilitas,246,31 December 2022,DATE_HEADER_PAIR
197,CTTH,2022,Q4,total_liabilities,7.173171e+08,CTTH_2022_Q4_FinancialStatement-2022-Tahunan-A...,1210000,Jumlah liabilitas,246,31 December 2022,DATE_HEADER_PAIR
198,DPNS,2021,Q4,cash,3.896022e+06,DPNS_2021_Q4_FS.xlsx,1210000,Kas dan setara kas,7,31 December 2021,DATE_HEADER_PAIR


In [35]:
# CELL 23D

source_type_df = (
    multi_file_conflict_detail
    .copy()
)

source_type_df[
    "source_file_type"
] = "OTHER"

short_fs_mask = (
    source_type_df[
        "source_file"
    ]
    .str.match(
        r"^[A-Za-z0-9]+_\d{4}_Q[1-4]_FS\.xlsx$",
        case=False,
        na=False
    )
)

financial_statement_mask = (
    source_type_df[
        "source_file"
    ]
    .str.contains(
        "FinancialStatement",
        case=False,
        na=False
    )
)

source_type_df.loc[
    short_fs_mask,
    "source_file_type"
] = "SHORT_FS"

source_type_df.loc[
    financial_statement_mask,
    "source_file_type"
] = "FINANCIAL_STATEMENT"


source_type_summary = (
    source_type_df[
        "source_file_type"
    ]
    .value_counts()
    .reset_index()
)

source_type_summary.columns = [
    "source_file_type",
    "rows"
]

display(
    source_type_summary
)

,source_file_type,rows
0,FINANCIAL_STATEMENT,537
1,SHORT_FS,531


In [36]:
# CELL 23E

conflict_source_type_summary = (
    source_type_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )[
        "source_file_type"
    ]
    .agg(
        lambda x: " + ".join(
            sorted(
                set(x)
            )
        )
    )
    .reset_index(
        name="source_type_combination"
    )
)

source_combination_summary = (
    conflict_source_type_summary[
        "source_type_combination"
    ]
    .value_counts()
    .reset_index()
)

source_combination_summary.columns = [
    "source_type_combination",
    "conflict_groups"
]

display(
    source_combination_summary
)

,source_type_combination,conflict_groups
0,FINANCIAL_STATEMENT + SHORT_FS,531


In [37]:
# CELL 23F

repeated_value_audit = (
    source_type_df
    .groupby(
        [
            "source_file_type",
            "metric",
            "candidate_value"
        ],
        dropna=False
    )
    .agg(
        ticker_count=(
            "ticker",
            "nunique"
        ),

        report_count=(
            "source_file",
            "nunique"
        )
    )
    .reset_index()
)

repeated_value_audit = (
    repeated_value_audit[
        repeated_value_audit[
            "ticker_count"
        ] > 1
    ]
    .sort_values(
        [
            "ticker_count",
            "report_count"
        ],
        ascending=[
            False,
            False
        ]
    )
)

display(
    repeated_value_audit.head(100)
)

,source_file_type,metric,candidate_value,ticker_count,report_count
2,FINANCIAL_STATEMENT,cash,5.113822e+08,44,44
10,FINANCIAL_STATEMENT,gross_profit,5.349147e+08,44,44
23,FINANCIAL_STATEMENT,operating_cash_flow,4.796918e+08,44,44
26,FINANCIAL_STATEMENT,revenue,9.081420e+08,44,44
36,FINANCIAL_STATEMENT,total_assets,1.286625e+09,44,44
45,FINANCIAL_STATEMENT,total_liabilities,7.173171e+08,44,44
51,SHORT_FS,cash,1.619616e+06,44,44
59,SHORT_FS,gross_profit,3.822117e+06,44,44
63,SHORT_FS,operating_cash_flow,1.835397e+06,44,44
70,SHORT_FS,revenue,2.182859e+07,44,44


In [ ]:
# CELL 23G

source_type_metric_summary = (
    source_type_df
    .groupby(
        [
            "source_file_type",
            "metric"
        ]
    )
    .agg(
        rows=(
            "candidate_value",
            "size"
        ),

        unique_values=(
            "candidate_value",
            "nunique"
        ),

        tickers=(
            "ticker",
            "nunique"
        ),

        source_files=(
            "source_file",
            "nunique"
        )
    )
    .reset_index()
)

display(
    source_type_metric_summary
)

,source_file_type,metric,rows,unique_values,tickers,source_files
0,FINANCIAL_STATEMENT,cash,89,8,46,89
1,FINANCIAL_STATEMENT,gross_profit,89,8,46,89
2,FINANCIAL_STATEMENT,operating_cash_flow,90,9,47,90
3,FINANCIAL_STATEMENT,revenue,89,8,46,89
4,FINANCIAL_STATEMENT,total_assets,90,9,47,90
5,FINANCIAL_STATEMENT,total_liabilities,90,9,47,90
6,SHORT_FS,cash,88,5,46,88
7,SHORT_FS,gross_profit,88,5,46,88
8,SHORT_FS,operating_cash_flow,89,6,47,89
9,SHORT_FS,revenue,88,5,46,88


In [39]:
# CELL 23H

conflict_source_paths = (
    source_type_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path",
            "source_file_type"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique conflict source files:",
    conflict_source_paths["source_file"].nunique()
)

print(
    "Unique source paths:",
    conflict_source_paths["source_path"].nunique()
)

display(
    conflict_source_paths.head(100)
)

Unique conflict source files: 179
Unique source paths: 179


,ticker,year,quarter,source_file,source_path,source_file_type
0,AVIA,2021,Q4,AVIA_2021_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,SHORT_FS
1,AVIA,2021,Q4,AVIA_2021_Q4_FinancialStatement-2021-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,FINANCIAL_STATEMENT
2,AVIA,2022,Q4,AVIA_2022_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,SHORT_FS
3,AVIA,2022,Q4,AVIA_2022_Q4_FinancialStatement-2022-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,FINANCIAL_STATEMENT
4,BFIN,2021,Q4,BFIN_2021_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,SHORT_FS
...,...,...,...,...,...,...
95,MEGA,2021,Q4,MEGA_2021_Q4_FinancialStatement-2021-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,FINANCIAL_STATEMENT
96,MEGA,2022,Q4,MEGA_2022_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,SHORT_FS
97,MEGA,2022,Q4,MEGA_2022_Q4_FinancialStatement-2022-Tahunan-A...,data\idx_financial_statements\Financial_Statem...,FINANCIAL_STATEMENT
98,MERK,2021,Q4,MERK_2021_Q4_FS.xlsx,data\idx_financial_statements\Financial_Statem...,SHORT_FS


In [40]:
path_reuse_audit = (
    conflict_source_paths
    .groupby(
        "source_path",
        dropna=False
    )
    .agg(
        ticker_count=("ticker", "nunique"),
        file_name_count=("source_file", "nunique")
    )
    .reset_index()
)

suspicious_path_reuse = (
    path_reuse_audit[
        path_reuse_audit["ticker_count"] > 1
    ]
    .sort_values(
        "ticker_count",
        ascending=False
    )
)

print(
    "Source paths reused across different tickers:",
    len(suspicious_path_reuse)
)

display(
    suspicious_path_reuse.head(100)
)

Source paths reused across different tickers: 0


,source_path,ticker_count,file_name_count


In [42]:
# CELL 23I - FIXED

source_metric_fingerprint = (
    source_type_df
    .pivot_table(
        index=[
            "ticker",
            "year",
            "quarter",
            "source_file",
            "source_path",
            "source_file_type"
        ],
        columns="metric",
        values="candidate_value",
        aggfunc="first"
    )
    .reset_index()
)

METRIC_COLUMNS = [
    "cash",
    "gross_profit",
    "operating_cash_flow",
    "revenue",
    "total_assets",
    "total_liabilities"
]

for col in METRIC_COLUMNS:
    if col not in source_metric_fingerprint.columns:
        source_metric_fingerprint[col] = np.nan


fingerprint_value_df = (
    source_metric_fingerprint[
        METRIC_COLUMNS
    ]
    .copy()
)

fingerprint_value_df = (
    fingerprint_value_df
    .fillna("<MISSING>")
    .astype(str)
)

source_metric_fingerprint[
    "metric_fingerprint"
] = (
    fingerprint_value_df
    .agg(
        "|".join,
        axis=1
    )
)


fingerprint_summary = (
    source_metric_fingerprint
    .groupby(
        "metric_fingerprint",
        dropna=False
    )
    .agg(
        tickers=(
            "ticker",
            "nunique"
        ),

        source_files=(
            "source_file",
            "nunique"
        ),

        file_types=(
            "source_file_type",
            lambda x:
                " + ".join(
                    sorted(
                        {
                            str(value)
                            for value in x
                            if pd.notna(value)
                        }
                    )
                )
        )
    )
    .reset_index()
    .sort_values(
        [
            "tickers",
            "source_files"
        ],
        ascending=[
            False,
            False
        ]
    )
)

display(
    fingerprint_summary.head(50)
)

,metric_fingerprint,tickers,source_files,file_types
3,1619616.0|3822117.0|1835397.0|21828591.0|29249...,44,45,FINANCIAL_STATEMENT + SHORT_FS
10,511382167.0|534914660.0|479691758.0|908142046....,44,44,FINANCIAL_STATEMENT
9,3896022.0|4830014.0|4895119.0|24322048.0|30399...,31,31,SHORT_FS
5,18722391189.0|77326783295.0|-28394083403.0|168...,30,30,FINANCIAL_STATEMENT
8,3874116.0|3611195.0|4138836.0|18014469.0|29694...,7,7,SHORT_FS
1,14470277252.0|27869550385.0|-22649188382.0|715...,5,5,FINANCIAL_STATEMENT
7,2532318.0|2214032.0|2399195.0|10832079.0|28689...,5,5,SHORT_FS
2,15541625679.0|57957556926.0|7203480979.0|11740...,4,4,FINANCIAL_STATEMENT
0,1352234305.0|27993452492.0|-4153774738.0|28326...,3,3,FINANCIAL_STATEMENT
4,1810842.0|932185.0|1004935.0|5035167.0|2846901...,1,1,SHORT_FS


In [43]:
# CELL 23J

from pathlib import Path
import hashlib


def sha256_file(
    file_path,
    chunk_size=1024 * 1024
):
    hasher = hashlib.sha256()

    with open(
        file_path,
        "rb"
    ) as file:

        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            hasher.update(
                chunk
            )

    return hasher.hexdigest()


PROJECT_ROOT = Path.cwd()


def resolve_conflict_path(
    source_path
):
    if pd.isna(source_path):
        return None

    path = Path(
        str(source_path)
    )

    if path.exists():
        return path

    relative_path = (
        PROJECT_ROOT
        /
        path
    )

    if relative_path.exists():
        return relative_path

    return None


conflict_file_hash_df = (
    conflict_source_paths
    .copy()
)

conflict_file_hash_df[
    "resolved_path"
] = (
    conflict_file_hash_df[
        "source_path"
    ]
    .apply(
        resolve_conflict_path
    )
)


print(
    "Files:",
    len(conflict_file_hash_df)
)

print(
    "Resolved:",
    conflict_file_hash_df[
        "resolved_path"
    ].notna().sum()
)

print(
    "Not resolved:",
    conflict_file_hash_df[
        "resolved_path"
    ].isna().sum()
)

Files: 179
Resolved: 179
Not resolved: 0


In [44]:
conflict_file_hash_df[
    "sha256"
] = (
    conflict_file_hash_df[
        "resolved_path"
    ]
    .apply(
        lambda path:
            sha256_file(path)
            if path is not None
            else None
    )
)

print(
    "Hashed files:",
    conflict_file_hash_df[
        "sha256"
    ].notna().sum()
)

Hashed files: 179


In [45]:
# CELL 23K

duplicate_hash_summary = (
    conflict_file_hash_df[
        conflict_file_hash_df[
            "sha256"
        ].notna()
    ]
    .groupby(
        "sha256"
    )
    .agg(
        file_count=(
            "source_file",
            "nunique"
        ),

        ticker_count=(
            "ticker",
            "nunique"
        ),

        source_types=(
            "source_file_type",
            lambda x:
                " + ".join(
                    sorted(
                        {
                            str(value)
                            for value in x
                            if pd.notna(value)
                        }
                    )
                )
        )
    )
    .reset_index()
)

duplicate_hash_summary = (
    duplicate_hash_summary[
        duplicate_hash_summary[
            "ticker_count"
        ] > 1
    ]
    .sort_values(
        [
            "ticker_count",
            "file_count"
        ],
        ascending=[
            False,
            False
        ]
    )
)

print(
    "Hashes shared across different tickers:",
    len(duplicate_hash_summary)
)

display(
    duplicate_hash_summary.head(100)
)

Hashes shared across different tickers: 9


,sha256,file_count,ticker_count,source_types
2,1b5f3fff5d1f7ffe62a48236b5e7a9546c20027ca4e780...,45,44,FINANCIAL_STATEMENT + SHORT_FS
6,92a7d3bb7b79619cec5725f888091fef2edda934163c58...,44,44,FINANCIAL_STATEMENT
7,966101ef0daf47728dc0fc975403cc16ca60be4ab06f93...,31,31,SHORT_FS
1,0ee4968f6a0aa570f3c30750b69ce4c1fb07fe08398773...,30,30,FINANCIAL_STATEMENT
12,f8e13afc80c729e85135ea91d873d75aa63d1a151cded9...,7,7,SHORT_FS
0,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...,5,5,SHORT_FS
13,faa4072594ce6997827382a01a5fa55fc9d15854a09067...,5,5,FINANCIAL_STATEMENT
9,a4078c45c47fcc8622c1f4b1c1e4331fbf2a6aa4d4dafa...,4,4,FINANCIAL_STATEMENT
5,4018ffee76f2fe2b4a3e1717b2d7f1b431ebab501e05f2...,3,3,FINANCIAL_STATEMENT


In [46]:
duplicate_hash_details = (
    conflict_file_hash_df
    .merge(
        duplicate_hash_summary[
            [
                "sha256"
            ]
        ],
        on="sha256",
        how="inner"
    )
    .sort_values(
        [
            "sha256",
            "ticker",
            "source_file"
        ]
    )
)

display(
    duplicate_hash_details[
        [
            "ticker",
            "year",
            "quarter",
            "source_file_type",
            "source_file",
            "source_path",
            "sha256"
        ]
    ].head(300)
)

,ticker,year,quarter,source_file_type,source_file,source_path,sha256
139,SMAR,2021,Q2,SHORT_FS,SMAR_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...
149,SMRU,2021,Q2,SHORT_FS,SMRU_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...
155,SPMA,2021,Q2,SHORT_FS,SPMA_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...
159,SULI,2021,Q2,SHORT_FS,SULI_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...
163,TMPO,2021,Q2,SHORT_FS,TMPO_2021_Q2_FS.xlsx,data\idx_financial_statements\Financial_Statem...,0c7298a8c08751ccffa02c396b576248384f0764c0a78f...
...,...,...,...,...,...,...,...
140,SMAR,2021,Q2,FINANCIAL_STATEMENT,SMAR_2021_Q2_FinancialStatement-2021-II-ABBA.xlsx,data\idx_financial_statements\Financial_Statem...,faa4072594ce6997827382a01a5fa55fc9d15854a09067...
150,SMRU,2021,Q2,FINANCIAL_STATEMENT,SMRU_2021_Q2_FinancialStatement-2021-II-ABBA.xlsx,data\idx_financial_statements\Financial_Statem...,faa4072594ce6997827382a01a5fa55fc9d15854a09067...
156,SPMA,2021,Q2,FINANCIAL_STATEMENT,SPMA_2021_Q2_FinancialStatement-2021-II-ABBA.xlsx,data\idx_financial_statements\Financial_Statem...,faa4072594ce6997827382a01a5fa55fc9d15854a09067...
160,SULI,2021,Q2,FINANCIAL_STATEMENT,SULI_2021_Q2_FinancialStatement-2021-II-ABBA.xlsx,data\idx_financial_statements\Financial_Statem...,faa4072594ce6997827382a01a5fa55fc9d15854a09067...


In [47]:
# CELL 23L

suspicious_hashes = set(
    duplicate_hash_summary[
        "sha256"
    ]
    .dropna()
)

print(
    "Suspicious duplicate hashes:",
    len(suspicious_hashes)
)

conflict_file_hash_df[
    "is_suspicious_duplicate"
] = (
    conflict_file_hash_df[
        "sha256"
    ]
    .isin(
        suspicious_hashes
    )
)

display(
    conflict_file_hash_df[
        "is_suspicious_duplicate"
    ]
    .value_counts()
)

Suspicious duplicate hashes: 9


is_suspicious_duplicate
True     174
False      5
Name: count, dtype: int64

In [51]:
# CELL 23M - FIXED

suspicious_file_lookup = (
    conflict_file_hash_df[
        conflict_file_hash_df[
            "is_suspicious_duplicate"
        ]
    ]
    [
        [
            "source_file",
            "source_path"
        ]
    ]
    .drop_duplicates()
    .copy()
)

suspicious_file_lookup[
    "suspicious_duplicate"
] = True


current_with_quality_df = (
    current_df
    .merge(
        suspicious_file_lookup,
        on=[
            "source_file",
            "source_path"
        ],
        how="left"
    )
)

current_with_quality_df[
    "suspicious_duplicate"
] = (
    current_with_quality_df[
        "suspicious_duplicate"
    ]
    .fillna(False)
    .astype(bool)
)

print(
    current_with_quality_df[
        "suspicious_duplicate"
    ].dtype
)

display(
    current_with_quality_df[
        "suspicious_duplicate"
    ]
    .value_counts(
        dropna=False
    )
)

bool


suspicious_duplicate
False    88665
True      1044
Name: count, dtype: int64

In [52]:
# CELL 23N - FIXED

conflict_quality_detail_df = (
    current_with_quality_df
    .merge(
        conflict_groups_df[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

conflict_quality_detail_df[
    "clean_candidate_value"
] = (
    conflict_quality_detail_df[
        "candidate_value"
    ]
    .where(
        ~conflict_quality_detail_df[
            "suspicious_duplicate"
        ]
    )
)

conflict_quality_audit = (
    conflict_quality_detail_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        total_candidates=(
            "candidate_value",
            "size"
        ),

        clean_candidates=(
            "suspicious_duplicate",
            lambda x: (~x).sum()
        ),

        suspicious_candidates=(
            "suspicious_duplicate",
            "sum"
        ),

        clean_unique_values=(
            "clean_candidate_value",
            "nunique"
        )
    )
    .reset_index()
)

conflict_quality_summary = (
    conflict_quality_audit[
        [
            "clean_candidates",
            "clean_unique_values"
        ]
    ]
    .value_counts()
    .reset_index(
        name="groups"
    )
    .sort_values(
        "groups",
        ascending=False
    )
)

display(
    conflict_quality_summary
)

,clean_candidates,clean_unique_values,groups
0,0,0,516
1,2,2,19
2,1,1,6
3,3,2,2
4,11,6,1
5,5,5,1


In [53]:
display(
    conflict_quality_audit.head(100)
)

,ticker,year,quarter,metric,total_candidates,clean_candidates,suspicious_candidates,clean_unique_values
0,ASHA,2024,Q2,revenue,11,11,0,6
1,ASHA,2024,Q3,revenue,5,5,0,5
2,AVIA,2021,Q4,cash,2,0,2,0
3,AVIA,2021,Q4,gross_profit,2,0,2,0
4,AVIA,2021,Q4,operating_cash_flow,2,0,2,0
...,...,...,...,...,...,...,...,...
95,CTTH,2022,Q4,cash,2,0,2,0
96,CTTH,2022,Q4,gross_profit,2,0,2,0
97,CTTH,2022,Q4,operating_cash_flow,2,0,2,0
98,CTTH,2022,Q4,revenue,2,0,2,0


In [54]:
# CELL 23O - AUDIT 23 UNRESOLVED CLEAN CONFLICTS

unresolved_clean_conflicts = (
    conflict_quality_audit[
        conflict_quality_audit[
            "clean_unique_values"
        ] > 1
    ]
    .copy()
)

print(
    "Unresolved clean conflict groups:",
    len(unresolved_clean_conflicts)
)

unresolved_clean_detail_df = (
    conflict_quality_detail_df
    .merge(
        unresolved_clean_conflicts[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
)

unresolved_clean_detail_df = (
    unresolved_clean_detail_df[
        ~unresolved_clean_detail_df[
            "suspicious_duplicate"
        ]
    ]
    .copy()
)

unresolved_clean_detail_df = (
    unresolved_clean_detail_df
    .sort_values(
        GROUP_KEYS
        +
        [
            "source_file",
            "source_sheet",
            "candidate_value"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    unresolved_clean_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "source_sheet",
            "source_label",
            "row_number",
            "period_header",
            "mapping_status"
        ]
    ].head(300)
)

Unresolved clean conflict groups: 23


,ticker,year,quarter,metric,candidate_value,source_file,source_sheet,source_label,row_number,period_header,mapping_status
0,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1311000,Sales and revenue,6,CurrentYearDuration,EXPLICIT_TOKEN
1,ASHA,2024,Q2,revenue,2.317741e+09,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,20,CurrentYearDuration,EXPLICIT_TOKEN
2,ASHA,2024,Q2,revenue,1.031772e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,19,CurrentYearDuration,EXPLICIT_TOKEN
3,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,30,CurrentYearDuration,EXPLICIT_TOKEN
4,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,31,CurrentYearDuration,EXPLICIT_TOKEN
5,ASHA,2024,Q2,revenue,2.317741e+09,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,8,CurrentYearDuration,EXPLICIT_TOKEN
6,ASHA,2024,Q2,revenue,2.052546e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,19,CurrentYearDuration,EXPLICIT_TOKEN
7,ASHA,2024,Q2,revenue,2.052546e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,30,CurrentYearDuration,EXPLICIT_TOKEN
8,ASHA,2024,Q2,revenue,8.265174e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,7,CurrentYearDuration,EXPLICIT_TOKEN
9,ASHA,2024,Q2,revenue,8.496948e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,18,CurrentYearDuration,EXPLICIT_TOKEN


In [55]:
# CELL 23P - SUMMARIZE UNRESOLVED CLEAN CONFLICTS

unresolved_pattern_summary = (
    unresolved_clean_detail_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        clean_rows=(
            "candidate_value",
            "size"
        ),

        unique_values=(
            "candidate_value",
            "nunique"
        ),

        source_files=(
            "source_file",
            "nunique"
        ),

        source_sheets=(
            "source_sheet",
            "nunique"
        ),

        source_labels=(
            "source_label",
            "nunique"
        )
    )
    .reset_index()
)

display(
    unresolved_pattern_summary
)

print(
    "Unresolved groups:",
    len(unresolved_pattern_summary)
)

,ticker,year,quarter,metric,clean_rows,unique_values,source_files,source_sheets,source_labels
0,ASHA,2024,Q2,revenue,11,6,1,3,2
1,ASHA,2024,Q3,revenue,5,5,1,3,2
2,BINA,2023,Q2,operating_cash_flow,2,2,2,1,1
3,BINA,2023,Q2,total_assets,2,2,2,1,1
4,BINA,2023,Q2,total_liabilities,2,2,2,1,1
5,GPSO,2025,Q1,revenue,2,2,1,2,2
6,HALO,2025,Q1,cash,2,2,1,2,1
7,MENN,2023,Q1,revenue,2,2,1,2,2
8,MLPL,2021,Q1,cash,2,2,2,1,1
9,MLPL,2021,Q1,gross_profit,2,2,2,2,1


Unresolved groups: 23


In [56]:
# CELL 23Q - AUTO-RESOLVABLE GROUPS ONLY

auto_resolvable_groups = (
    conflict_quality_audit[
        (
            conflict_quality_audit[
                "clean_candidates"
            ] >= 1
        )
        &
        (
            conflict_quality_audit[
                "clean_unique_values"
            ] == 1
        )
    ]
    .copy()
)

print(
    "Auto-resolvable conflict groups:",
    len(auto_resolvable_groups)
)

auto_resolvable_detail_df = (
    conflict_quality_detail_df
    .merge(
        auto_resolvable_groups[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
)

auto_resolvable_detail_df = (
    auto_resolvable_detail_df[
        ~auto_resolvable_detail_df[
            "suspicious_duplicate"
        ]
    ]
    .copy()
)

display(
    auto_resolvable_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "source_sheet",
            "source_label",
            "row_number"
        ]
    ].head(100)
)

Auto-resolvable conflict groups: 6


,ticker,year,quarter,metric,candidate_value,source_file,source_sheet,source_label,row_number
0,RIGS,2021,Q4,cash,2.372204e+08,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1210000,Kas dan setara kas,7
1,RIGS,2021,Q4,total_assets,1.036704e+09,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1210000,Jumlah aset,123
2,RIGS,2021,Q4,total_liabilities,6.798150e+08,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1210000,Jumlah liabilitas,232
3,RIGS,2021,Q4,revenue,1.021865e+09,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1311000,Sales and revenue,5
4,RIGS,2021,Q4,gross_profit,3.663568e+08,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1311000,Jumlah laba bruto,7
5,RIGS,2021,Q4,operating_cash_flow,3.645590e+08,RIGS_2021_Q4_FinancialStatement-2021-Tahunan-A...,1510000,Total net cash flows received from (used in) o...,37


In [57]:
# CELL 23R - DETAIL UNRESOLVED BY SOURCE STRUCTURE

unresolved_structure_df = (
    unresolved_clean_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "source_sheet",
            "source_label",
            "row_number",
            "period_header",
            "mapping_status"
        ]
    ]
    .copy()
)

display(
    unresolved_structure_df
    .sort_values(
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_file",
            "source_sheet",
            "row_number"
        ]
    )
)

,ticker,year,quarter,metric,candidate_value,source_file,source_sheet,source_label,row_number,period_header,mapping_status
0,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1311000,Sales and revenue,6,CurrentYearDuration,EXPLICIT_TOKEN
2,ASHA,2024,Q2,revenue,1.031772e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,19,CurrentYearDuration,EXPLICIT_TOKEN
1,ASHA,2024,Q2,revenue,2.317741e+09,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,20,CurrentYearDuration,EXPLICIT_TOKEN
3,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,30,CurrentYearDuration,EXPLICIT_TOKEN
4,ASHA,2024,Q2,revenue,1.054949e+11,ASHA_2024_Q2_FS.xlsx,1617000,PENJUALAN,31,CurrentYearDuration,EXPLICIT_TOKEN
8,ASHA,2024,Q2,revenue,8.265174e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,7,CurrentYearDuration,EXPLICIT_TOKEN
5,ASHA,2024,Q2,revenue,2.317741e+09,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,8,CurrentYearDuration,EXPLICIT_TOKEN
9,ASHA,2024,Q2,revenue,8.496948e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,18,CurrentYearDuration,EXPLICIT_TOKEN
6,ASHA,2024,Q2,revenue,2.052546e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,19,CurrentYearDuration,EXPLICIT_TOKEN
7,ASHA,2024,Q2,revenue,2.052546e+10,ASHA_2024_Q2_FS.xlsx,1618000,PENJUALAN,30,CurrentYearDuration,EXPLICIT_TOKEN


In [58]:
# CELL 23S - GROUP CONFLICT TYPE

unresolved_type_summary = (
    unresolved_pattern_summary
    .assign(
        conflict_type=np.select(
            [
                unresolved_pattern_summary[
                    "source_files"
                ] > 1,

                unresolved_pattern_summary[
                    "source_files"
                ].eq(1)
                &
                (
                    unresolved_pattern_summary[
                        "source_sheets"
                    ] > 1
                )
            ],
            [
                "MULTIPLE_FILES",
                "MULTIPLE_SHEETS_SAME_FILE"
            ],
            default="OTHER"
        )
    )
)

display(
    unresolved_type_summary[
        "conflict_type"
    ]
    .value_counts()
    .reset_index(
        name="groups"
    )
)

display(
    unresolved_type_summary
)

,conflict_type,groups
0,MULTIPLE_SHEETS_SAME_FILE,14
1,MULTIPLE_FILES,9


,ticker,year,quarter,metric,clean_rows,unique_values,source_files,source_sheets,source_labels,conflict_type
0,ASHA,2024,Q2,revenue,11,6,1,3,2,MULTIPLE_SHEETS_SAME_FILE
1,ASHA,2024,Q3,revenue,5,5,1,3,2,MULTIPLE_SHEETS_SAME_FILE
2,BINA,2023,Q2,operating_cash_flow,2,2,2,1,1,MULTIPLE_FILES
3,BINA,2023,Q2,total_assets,2,2,2,1,1,MULTIPLE_FILES
4,BINA,2023,Q2,total_liabilities,2,2,2,1,1,MULTIPLE_FILES
5,GPSO,2025,Q1,revenue,2,2,1,2,2,MULTIPLE_SHEETS_SAME_FILE
6,HALO,2025,Q1,cash,2,2,1,2,1,MULTIPLE_SHEETS_SAME_FILE
7,MENN,2023,Q1,revenue,2,2,1,2,2,MULTIPLE_SHEETS_SAME_FILE
8,MLPL,2021,Q1,cash,2,2,2,1,1,MULTIPLE_FILES
9,MLPL,2021,Q1,gross_profit,2,2,2,2,1,MULTIPLE_FILES


In [59]:
# CELL 23T - LABEL / SHEET PATTERN

label_sheet_conflict_summary = (
    unresolved_clean_detail_df
    .groupby(
        [
            "metric",
            "source_sheet",
            "source_label"
        ],
        dropna=False
    )
    .agg(
        rows=(
            "candidate_value",
            "size"
        ),
        tickers=(
            "ticker",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        [
            "metric",
            "rows"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    label_sheet_conflict_summary
)

,metric,source_sheet,source_label,rows,tickers
0,cash,1210000,Kas dan setara kas,2,1
1,cash,1210000,Kas dan setara kas,1,1
2,cash,1610000,Kas dan setara kas,1,1
3,gross_profit,1311000,Jumlah laba bruto,1,1
4,gross_profit,1321000,Jumlah laba bruto,1,1
5,operating_cash_flow,1510000,Total net cash flows received from (used in) o...,2,1
6,operating_cash_flow,4510000,Total net cash flows received from (used in) o...,2,1
7,revenue,1311000,Sales and revenue,12,6
11,revenue,1617000,Penjualan,8,4
12,revenue,1618000,PENJUALAN,8,1


In [64]:
# CELL 23U - PREFERRED PRIMARY STATEMENT SHEETS

PREFERRED_SHEETS = {
    "cash": {
        "1210000",
        "4220000",
        "3210000",
    },

    "total_assets": {
        "1210000",
        "4220000",
        "3210000",
    },

    "total_liabilities": {
        "1210000",
        "4220000",
        "3210000",
    },

    "revenue": {
        "1311000",
        "1312000",
        "1321000",
        "4312000",
        "3312000",
    },

    "gross_profit": {
        "1311000",
        "1321000",
        "3312000",
    },

    "operating_cash_flow": {
        "1510000",
        "4510000",
        "3510000",
    },
}

In [65]:
# CELL 23V - RESOLVE SAME-FILE CONFLICTS BY PRIMARY SHEET

same_file_conflict_groups = (
    unresolved_type_summary[
        unresolved_type_summary[
            "conflict_type"
        ].eq(
            "MULTIPLE_SHEETS_SAME_FILE"
        )
    ]
    [GROUP_KEYS]
    .copy()
)

same_file_conflict_detail_df = (
    unresolved_clean_detail_df
    .merge(
        same_file_conflict_groups,
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

same_file_conflict_detail_df[
    "is_preferred_sheet"
] = (
    same_file_conflict_detail_df
    .apply(
        lambda row:
            str(row["source_sheet"])
            in
            PREFERRED_SHEETS.get(
                row["metric"],
                set()
            ),
        axis=1
    )
)

display(
    same_file_conflict_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_sheet",
            "source_label",
            "is_preferred_sheet",
            "source_file"
        ]
    ]
    .sort_values(
        GROUP_KEYS
        +
        [
            "is_preferred_sheet"
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False
        ]
    )
)

,ticker,year,quarter,metric,candidate_value,source_sheet,source_label,is_preferred_sheet,source_file
0,ASHA,2024,Q2,revenue,1.054949e+11,1311000,Sales and revenue,True,ASHA_2024_Q2_FS.xlsx
1,ASHA,2024,Q2,revenue,2.317741e+09,1617000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
2,ASHA,2024,Q2,revenue,1.031772e+11,1617000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
3,ASHA,2024,Q2,revenue,1.054949e+11,1617000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
4,ASHA,2024,Q2,revenue,1.054949e+11,1617000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
5,ASHA,2024,Q2,revenue,2.317741e+09,1618000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
6,ASHA,2024,Q2,revenue,2.052546e+10,1618000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
7,ASHA,2024,Q2,revenue,2.052546e+10,1618000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
8,ASHA,2024,Q2,revenue,8.265174e+10,1618000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx
9,ASHA,2024,Q2,revenue,8.496948e+10,1618000,PENJUALAN,False,ASHA_2024_Q2_FS.xlsx


In [66]:
# CELL 23W - VALIDATE PRIMARY-SHEET RESOLUTION

preferred_same_file_df = (
    same_file_conflict_detail_df[
        same_file_conflict_detail_df[
            "is_preferred_sheet"
        ]
    ]
    .copy()
)

preferred_same_file_summary = (
    preferred_same_file_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        preferred_rows=(
            "candidate_value",
            "size"
        ),

        preferred_unique_values=(
            "candidate_value",
            "nunique"
        ),

        preferred_sheets=(
            "source_sheet",
            "nunique"
        )
    )
    .reset_index()
)

display(
    preferred_same_file_summary
)

print(
    "Same-file groups:",
    len(same_file_conflict_groups)
)

print(
    "Groups with exactly one preferred unique value:",
    (
        preferred_same_file_summary[
            "preferred_unique_values"
        ]
        .eq(1)
        .sum()
    )
)

,ticker,year,quarter,metric,preferred_rows,preferred_unique_values,preferred_sheets
0,ASHA,2024,Q2,revenue,1,1,1
1,ASHA,2024,Q3,revenue,1,1,1
2,GPSO,2025,Q1,revenue,1,1,1
3,HALO,2025,Q1,cash,1,1,1
4,MENN,2023,Q1,revenue,1,1,1
5,RAFI,2024,Q3,revenue,1,1,1
6,RANC,2022,Q4,revenue,1,1,1
7,RANC,2023,Q1,revenue,1,1,1
8,RANC,2023,Q2,revenue,1,1,1
9,SDMU,2023,Q1,revenue,1,1,1


Same-file groups: 14
Groups with exactly one preferred unique value: 14


In [68]:
# CELL 23X - FIND THE 1 UNRESOLVED SAME-FILE GROUP

resolved_same_file_keys = (
    preferred_same_file_summary[
        preferred_same_file_summary[
            "preferred_unique_values"
        ].eq(1)
    ]
    [GROUP_KEYS]
    .copy()
)

unresolved_same_file_groups = (
    same_file_conflict_groups
    .merge(
        resolved_same_file_keys,
        on=GROUP_KEYS,
        how="left",
        indicator=True
    )
)

unresolved_same_file_groups = (
    unresolved_same_file_groups[
        unresolved_same_file_groups[
            "_merge"
        ].eq("left_only")
    ]
    [GROUP_KEYS]
    .copy()
)

print(
    "Remaining unresolved same-file groups:",
    len(unresolved_same_file_groups)
)

remaining_same_file_detail = (
    same_file_conflict_detail_df
    .merge(
        unresolved_same_file_groups,
        on=GROUP_KEYS,
        how="inner"
    )
)

display(
    remaining_same_file_detail[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_sheet",
            "source_label",
            "row_number",
            "source_file",
            "period_header",
            "mapping_status"
        ]
    ]
)

Remaining unresolved same-file groups: 0


,ticker,year,quarter,metric,candidate_value,source_sheet,source_label,row_number,source_file,period_header,mapping_status


In [69]:
# CELL 23Y - MULTIPLE FILE CONFLICTS

multiple_file_groups = (
    unresolved_type_summary[
        unresolved_type_summary[
            "conflict_type"
        ].eq(
            "MULTIPLE_FILES"
        )
    ]
    [GROUP_KEYS]
    .copy()
)

multiple_file_detail_df = (
    unresolved_clean_detail_df
    .merge(
        multiple_file_groups,
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

print(
    "Multiple-file groups:",
    len(multiple_file_groups)
)

display(
    multiple_file_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "source_sheet",
            "source_label",
            "period_header",
            "mapping_status"
        ]
    ]
    .sort_values(
        GROUP_KEYS
        +
        [
            "source_file"
        ]
    )
)

Multiple-file groups: 9


,ticker,year,quarter,metric,candidate_value,source_file,source_sheet,source_label,period_header,mapping_status
0,BINA,2023,Q2,operating_cash_flow,-9.075080e+05,BINA_2023_Q2_FS.xlsx,4510000,Total net cash flows received from (used in) o...,31 March 2023,DATE_HEADER_PAIR
1,BINA,2023,Q2,operating_cash_flow,-2.241139e+06,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,4510000,Total net cash flows received from (used in) o...,30 June 2023,DATE_HEADER_PAIR
2,BINA,2023,Q2,total_assets,2.141983e+07,BINA_2023_Q2_FS.xlsx,4220000,Jumlah aset,31 March 2023,DATE_HEADER_PAIR
3,BINA,2023,Q2,total_assets,2.229680e+07,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,4220000,Jumlah aset,30 June 2023,DATE_HEADER_PAIR
4,BINA,2023,Q2,total_liabilities,1.804821e+07,BINA_2023_Q2_FS.xlsx,4220000,Jumlah liabilitas,31 March 2023,DATE_HEADER_PAIR
5,BINA,2023,Q2,total_liabilities,1.882235e+07,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,4220000,Jumlah liabilitas,30 June 2023,DATE_HEADER_PAIR
6,MLPL,2021,Q1,cash,1.810842e+06,MLPL_2021_Q1_FS.xlsx,1210000,Kas dan setara kas,31 March 2021,DATE_HEADER_PAIR
7,MLPL,2021,Q1,cash,8.623326e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,1210000,Kas dan setara kas,31 March 2021,DATE_HEADER_PAIR
8,MLPL,2021,Q1,gross_profit,9.321850e+05,MLPL_2021_Q1_FS.xlsx,1321000,Jumlah laba bruto,31 March 2021,DATE_HEADER_PAIR
9,MLPL,2021,Q1,gross_profit,5.041608e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,1311000,Jumlah laba bruto,31 March 2021,DATE_HEADER_PAIR


In [71]:
# CELL 23Z0 - DATE HELPERS

import re
from datetime import datetime


MONTH_MAP = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,

    "januari": 1,
    "februari": 2,
    "maret": 3,
    "april": 4,
    "mei": 5,
    "juni": 6,
    "juli": 7,
    "agustus": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "desember": 12,
}


def parse_header_date(value):

    if value is None:
        return None

    if isinstance(value, datetime):
        return value.date()

    text = str(value).strip().lower()

    match = re.fullmatch(
        r"(\d{1,2})\s+([a-zA-Z]+)\s+(\d{4})",
        text
    )

    if not match:
        return None

    day = int(match.group(1))
    month_text = match.group(2)
    year = int(match.group(3))

    month = MONTH_MAP.get(
        month_text
    )

    if month is None:
        return None

    try:
        return datetime(
            year,
            month,
            day
        ).date()

    except ValueError:
        return None


def expected_report_date(
    year,
    quarter
):

    year = int(year)

    quarter_map = {
        "Q1": (3, 31),
        "Q2": (6, 30),
        "Q3": (9, 30),
        "Q4": (12, 31),
    }

    month, day = quarter_map[
        str(quarter).upper()
    ]

    return datetime(
        year,
        month,
        day
    ).date()

In [72]:
# CELL 23Z - PERIOD DATE MATCHING

multiple_file_detail_df[
    "parsed_period_date"
] = (
    multiple_file_detail_df[
        "period_header"
    ]
    .apply(
        parse_header_date
    )
)

multiple_file_detail_df[
    "expected_period_date"
] = (
    multiple_file_detail_df
    .apply(
        lambda row:
            expected_report_date(
                row["year"],
                row["quarter"]
            ),
        axis=1
    )
)

display(
    multiple_file_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "period_header",
            "parsed_period_date",
            "expected_period_date"
        ]
    ]
    .sort_values(
        GROUP_KEYS
        +
        [
            "source_file"
        ]
    )
)

,ticker,year,quarter,metric,candidate_value,source_file,period_header,parsed_period_date,expected_period_date
0,BINA,2023,Q2,operating_cash_flow,-9.075080e+05,BINA_2023_Q2_FS.xlsx,31 March 2023,2023-03-31,2023-06-30
1,BINA,2023,Q2,operating_cash_flow,-2.241139e+06,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,30 June 2023,2023-06-30,2023-06-30
2,BINA,2023,Q2,total_assets,2.141983e+07,BINA_2023_Q2_FS.xlsx,31 March 2023,2023-03-31,2023-06-30
3,BINA,2023,Q2,total_assets,2.229680e+07,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,30 June 2023,2023-06-30,2023-06-30
4,BINA,2023,Q2,total_liabilities,1.804821e+07,BINA_2023_Q2_FS.xlsx,31 March 2023,2023-03-31,2023-06-30
5,BINA,2023,Q2,total_liabilities,1.882235e+07,BINA_2023_Q2_FinancialStatement-2023-II-BINA.xlsx,30 June 2023,2023-06-30,2023-06-30
6,MLPL,2021,Q1,cash,1.810842e+06,MLPL_2021_Q1_FS.xlsx,31 March 2021,2021-03-31,2021-03-31
7,MLPL,2021,Q1,cash,8.623326e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,31 March 2021,2021-03-31,2021-03-31
8,MLPL,2021,Q1,gross_profit,9.321850e+05,MLPL_2021_Q1_FS.xlsx,31 March 2021,2021-03-31,2021-03-31
9,MLPL,2021,Q1,gross_profit,5.041608e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,31 March 2021,2021-03-31,2021-03-31


In [74]:
# CELL 23AA - SAFE VERSION

multiple_file_detail_df[
    "exact_match_value"
] = np.where(
    multiple_file_detail_df[
        "is_exact_period_match"
    ],
    multiple_file_detail_df[
        "candidate_value"
    ],
    np.nan
)

multiple_file_period_summary = (
    multiple_file_detail_df
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        candidate_rows=(
            "candidate_value",
            "size"
        ),

        exact_match_rows=(
            "is_exact_period_match",
            "sum"
        ),

        exact_match_unique_values=(
            "exact_match_value",
            "nunique"
        )
    )
    .reset_index()
)

display(
    multiple_file_period_summary
)

,ticker,year,quarter,metric,candidate_rows,exact_match_rows,exact_match_unique_values
0,BINA,2023,Q2,operating_cash_flow,2,1,1
1,BINA,2023,Q2,total_assets,2,1,1
2,BINA,2023,Q2,total_liabilities,2,1,1
3,MLPL,2021,Q1,cash,2,2,2
4,MLPL,2021,Q1,gross_profit,2,2,2
5,MLPL,2021,Q1,operating_cash_flow,2,2,2
6,MLPL,2021,Q1,revenue,2,2,2
7,MLPL,2021,Q1,total_assets,2,2,2
8,MLPL,2021,Q1,total_liabilities,2,2,2


In [75]:
# CELL 23AB - MULTI-FILE AUTO RESOLUTION AUDIT

multi_file_auto_resolvable = (
    multiple_file_period_summary[
        (
            multiple_file_period_summary[
                "exact_match_rows"
            ] >= 1
        )
        &
        (
            multiple_file_period_summary[
                "exact_match_unique_values"
            ] == 1
        )
    ]
    .copy()
)

multi_file_still_unresolved = (
    multiple_file_period_summary[
        ~multiple_file_period_summary[
            GROUP_KEYS
        ]
        .apply(tuple, axis=1)
        .isin(
            multi_file_auto_resolvable[
                GROUP_KEYS
            ]
            .apply(tuple, axis=1)
        )
    ]
    .copy()
)

print(
    "Multi-file auto-resolvable:",
    len(multi_file_auto_resolvable)
)

print(
    "Multi-file still unresolved:",
    len(multi_file_still_unresolved)
)

display(
    multi_file_auto_resolvable
)

display(
    multi_file_still_unresolved
)

Multi-file auto-resolvable: 3
Multi-file still unresolved: 6


,ticker,year,quarter,metric,candidate_rows,exact_match_rows,exact_match_unique_values
0,BINA,2023,Q2,operating_cash_flow,2,1,1
1,BINA,2023,Q2,total_assets,2,1,1
2,BINA,2023,Q2,total_liabilities,2,1,1


,ticker,year,quarter,metric,candidate_rows,exact_match_rows,exact_match_unique_values
3,MLPL,2021,Q1,cash,2,2,2
4,MLPL,2021,Q1,gross_profit,2,2,2
5,MLPL,2021,Q1,operating_cash_flow,2,2,2
6,MLPL,2021,Q1,revenue,2,2,2
7,MLPL,2021,Q1,total_assets,2,2,2
8,MLPL,2021,Q1,total_liabilities,2,2,2


In [76]:
# CELL 23AC - AUDIT FILE TICKER IDENTITY

def extract_trailing_ticker_from_filename(filename):
    if pd.isna(filename):
        return None

    name = str(filename)

    match = re.search(
        r"-([A-Za-z0-9]+)\.xlsx$",
        name,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).upper()

    return None


mlpl_unresolved_detail_df = (
    multiple_file_detail_df
    .merge(
        multi_file_still_unresolved[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .copy()
)

mlpl_unresolved_detail_df[
    "filename_trailing_ticker"
] = (
    mlpl_unresolved_detail_df[
        "source_file"
    ]
    .apply(
        extract_trailing_ticker_from_filename
    )
)

mlpl_unresolved_detail_df[
    "ticker_matches_filename"
] = (
    mlpl_unresolved_detail_df[
        "filename_trailing_ticker"
    ].isna()
    |
    (
        mlpl_unresolved_detail_df[
            "filename_trailing_ticker"
        ]
        ==
        mlpl_unresolved_detail_df[
            "ticker"
        ].str.upper()
    )
)

display(
    mlpl_unresolved_detail_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "source_file",
            "filename_trailing_ticker",
            "ticker_matches_filename",
            "source_sheet",
            "period_header"
        ]
    ]
)

,ticker,year,quarter,metric,candidate_value,source_file,filename_trailing_ticker,ticker_matches_filename,source_sheet,period_header
0,MLPL,2021,Q1,cash,1.810842e+06,MLPL_2021_Q1_FS.xlsx,NaN,True,1210000,31 March 2021
1,MLPL,2021,Q1,cash,8.623326e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,AGAR,False,1210000,31 March 2021
2,MLPL,2021,Q1,gross_profit,9.321850e+05,MLPL_2021_Q1_FS.xlsx,NaN,True,1321000,31 March 2021
3,MLPL,2021,Q1,gross_profit,5.041608e+09,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,AGAR,False,1311000,31 March 2021
4,MLPL,2021,Q1,operating_cash_flow,1.004935e+06,MLPL_2021_Q1_FS.xlsx,NaN,True,1510000,31 March 2021
5,MLPL,2021,Q1,operating_cash_flow,-3.451840e+08,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,AGAR,False,1510000,31 March 2021
6,MLPL,2021,Q1,revenue,5.035167e+06,MLPL_2021_Q1_FS.xlsx,NaN,True,1321000,31 March 2021
7,MLPL,2021,Q1,revenue,5.573440e+10,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,AGAR,False,1311000,31 March 2021
8,MLPL,2021,Q1,total_assets,2.846902e+07,MLPL_2021_Q1_FS.xlsx,NaN,True,1210000,31 March 2021
9,MLPL,2021,Q1,total_assets,1.787385e+11,MLPL_2021_Q1_FinancialStatement-2021-I-AGAR.xlsx,AGAR,False,1210000,31 March 2021


In [77]:
# CELL 23AD - RESOLVE MULTI-FILE BY TICKER IDENTITY

mlpl_unresolved_detail_df[
    "identity_valid_candidate"
] = (
    mlpl_unresolved_detail_df[
        "ticker_matches_filename"
    ]
)

identity_resolution_summary = (
    mlpl_unresolved_detail_df
    .assign(
        identity_value=np.where(
            mlpl_unresolved_detail_df[
                "identity_valid_candidate"
            ],
            mlpl_unresolved_detail_df[
                "candidate_value"
            ],
            np.nan
        )
    )
    .groupby(
        GROUP_KEYS,
        dropna=False
    )
    .agg(
        valid_candidate_rows=(
            "identity_valid_candidate",
            "sum"
        ),

        valid_unique_values=(
            "identity_value",
            "nunique"
        )
    )
    .reset_index()
)

display(
    identity_resolution_summary
)

print(
    "Identity-resolvable groups:",
    (
        (
            identity_resolution_summary[
                "valid_candidate_rows"
            ] >= 1
        )
        &
        (
            identity_resolution_summary[
                "valid_unique_values"
            ] == 1
        )
    ).sum()
)

,ticker,year,quarter,metric,valid_candidate_rows,valid_unique_values
0,MLPL,2021,Q1,cash,1,1
1,MLPL,2021,Q1,gross_profit,1,1
2,MLPL,2021,Q1,operating_cash_flow,1,1
3,MLPL,2021,Q1,revenue,1,1
4,MLPL,2021,Q1,total_assets,1,1
5,MLPL,2021,Q1,total_liabilities,1,1


Identity-resolvable groups: 6


In [78]:
# CELL 23AE - BUILD RESOLVED CONFLICT ROWS

# 1. Auto-resolvable clean groups
resolved_auto_df = (
    auto_resolvable_detail_df
    .drop_duplicates(
        subset=GROUP_KEYS
    )
    .copy()
)

resolved_auto_df[
    "selection_status"
] = "RESOLVED_CLEAN_SINGLE_VALUE"


# 2. Same-file conflicts resolved by preferred sheet
resolved_same_file_df = (
    preferred_same_file_df
    .drop_duplicates(
        subset=GROUP_KEYS
    )
    .copy()
)

resolved_same_file_df[
    "selection_status"
] = "RESOLVED_PREFERRED_PRIMARY_SHEET"


# 3. Multi-file conflicts resolved by exact period match
resolved_exact_period_df = (
    multiple_file_detail_df[
        multiple_file_detail_df[
            "is_exact_period_match"
        ]
    ]
    .merge(
        multi_file_auto_resolvable[
            GROUP_KEYS
        ],
        on=GROUP_KEYS,
        how="inner"
    )
    .drop_duplicates(
        subset=GROUP_KEYS
    )
    .copy()
)

resolved_exact_period_df[
    "selection_status"
] = "RESOLVED_EXACT_PERIOD_MATCH"


# 4. MLPL identity-resolved groups
resolved_identity_df = (
    mlpl_unresolved_detail_df[
        mlpl_unresolved_detail_df[
            "identity_valid_candidate"
        ]
    ]
    .drop_duplicates(
        subset=GROUP_KEYS
    )
    .copy()
)

resolved_identity_df[
    "selection_status"
] = "RESOLVED_TICKER_IDENTITY"


print("Resolved auto:", len(resolved_auto_df))
print("Resolved same-file:", len(resolved_same_file_df))
print("Resolved exact-period:", len(resolved_exact_period_df))
print("Resolved identity:", len(resolved_identity_df))

Resolved auto: 6
Resolved same-file: 14
Resolved exact-period: 3
Resolved identity: 6


In [79]:
# CELL 23AF - COMBINE SAFE + RESOLVED

resolved_conflicts_df = pd.concat(
    [
        resolved_auto_df,
        resolved_same_file_df,
        resolved_exact_period_df,
        resolved_identity_df
    ],
    ignore_index=True,
    sort=False
)

print(
    "Resolved conflict groups total:",
    len(resolved_conflicts_df)
)

print(
    "Duplicate resolved groups:",
    resolved_conflicts_df
    .duplicated(
        subset=GROUP_KEYS
    )
    .sum()
)

Resolved conflict groups total: 29
Duplicate resolved groups: 0


In [80]:
resolved_conflicts_df = (
    resolved_conflicts_df
    .rename(
        columns={
            "candidate_value":
                "selected_value"
        }
    )
)

In [81]:
final_selected_df = pd.concat(
    [
        selected_current_df,
        resolved_conflicts_df
    ],
    ignore_index=True,
    sort=False
)

In [82]:
# CELL 23AG - FINAL VALIDATION

final_duplicate_count = (
    final_selected_df
    .duplicated(
        subset=GROUP_KEYS
    )
    .sum()
)

print(
    "Final selected rows:",
    len(final_selected_df)
)

print(
    "Duplicate final groups:",
    final_duplicate_count
)

print(
    "Expected rows:",
    len(selected_current_df)
    +
    len(resolved_conflicts_df)
)

assert final_duplicate_count == 0

Final selected rows: 88400
Duplicate final groups: 0
Expected rows: 88400


In [22]:
value_missing_summary = (
    mapped_df[
        mapped_df["period_type"]
        .eq("VALUE_MISSING")
    ]
    .groupby(
        "metric"
    )
    .size()
    .reset_index(
        name="value_missing_rows"
    )
    .sort_values(
        "value_missing_rows",
        ascending=False
    )
)

display(
    value_missing_summary
)

,metric,value_missing_rows
3,revenue,17485
0,cash,8052
1,gross_profit,44
2,operating_cash_flow,4
4,total_assets,4
5,total_liabilities,4


In [23]:
duplicate_final = (
    selected_current_df
    .duplicated(
        subset=GROUP_KEYS
    )
    .sum()
)

print(
    "Duplicate final metric groups:",
    duplicate_final
)

print(
    "Selected rows:",
    len(selected_current_df)
)

print(
    "Conflict groups:",
    len(conflict_groups_df)
)

assert duplicate_final == 0, (
    "Final dataset masih punya "
    "duplicate ticker/year/quarter/metric."
)

Duplicate final metric groups: 0
Selected rows: 88371
Conflict groups: 545


In [83]:
# FINAL SAVE - SELECTED METRICS

FINAL_OUTPUT_FILE = Path(
    "data/idx_financial_current_metrics_final.csv"
)

final_selected_df.to_csv(
    FINAL_OUTPUT_FILE,
    index=False
)

print(
    "Saved final selected metrics:",
    FINAL_OUTPUT_FILE
)

print(
    "Rows:",
    len(final_selected_df)
)

Saved final selected metrics: data\idx_financial_current_metrics_final.csv
Rows: 88400


In [25]:
conflict_detail_df.to_csv(
    CONFLICT_FILE,
    index=False
)

print(
    "Saved conflict details:",
    CONFLICT_FILE
)

print(
    "Rows:",
    len(conflict_detail_df)
)

Saved conflict details: data\idx_financial_current_metric_conflicts.csv
Rows: 1110
